# Retail Basket Association Matrix Circular Chart

Chord diagrams and circular relation graphs display multi-group relationships and transaction dependencies between categories. In retail market basket analysis, they map co-purchase associations between product categories. This notebook generates synthetic transaction associations across ten departments and visualizes correlation relationships on a circular coordinate node layout in Plotly.



In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Define 10 product categories
categories = [
    'Bakery', 'Dairy', 'Produce', 'Meat', 'Pantry',
    'Snacks', 'Beverages', 'Frozen', 'Deli', 'Bakeware'
]
n_items = len(categories)

# Generate symmetric correlation association matrix (co-purchase frequencies)
np.random.seed(135)
assoc_matrix = np.random.uniform(0.05, 0.65, (n_items, n_items))
assoc_matrix = (assoc_matrix + assoc_matrix.T) / 2.0 # Make symmetric
np.fill_diagonal(assoc_matrix, 0.0) # Zero self associations

df_assoc = pd.DataFrame(assoc_matrix, index=categories, columns=categories)
df_assoc



,Bakery,Dairy,Produce,Meat,Pantry,Snacks,Beverages,Frozen,Deli,Bakeware
Bakery,0.000000,0.260304,0.188524,0.589854,0.529461,0.284626,0.462289,0.516844,0.262980,0.399056
Dairy,0.260304,0.000000,0.211176,0.140733,0.239461,0.383556,0.518698,0.353854,0.315401,0.464639
Produce,0.188524,0.211176,0.000000,0.484145,0.330982,0.442319,0.324249,0.456962,0.482016,0.536433
Meat,0.589854,0.140733,0.484145,0.000000,0.186505,0.424347,0.620706,0.283981,0.459803,0.464214
Pantry,0.529461,0.239461,0.330982,0.186505,0.000000,0.228745,0.325621,0.453542,0.263466,0.350921
Snacks,0.284626,0.383556,0.442319,0.424347,0.228745,0.000000,0.239768,0.186191,0.350724,0.470407
Beverages,0.462289,0.518698,0.324249,0.620706,0.325621,0.239768,0.000000,0.618480,0.224971,0.408742
Frozen,0.516844,0.353854,0.456962,0.283981,0.453542,0.186191,0.618480,0.000000,0.538367,0.497501
Deli,0.262980,0.315401,0.482016,0.459803,0.263466,0.350724,0.224971,0.538367,0.000000,0.430425
Bakeware,0.399056,0.464639,0.536433,0.464214,0.350921,0.470407,0.408742,0.497501,0.430425,0.000000


## Circular Node Coordinates Layout

We distribute the 10 nodes evenly along a unit circle circumference to establish visual coordinates.



In [2]:
angles = np.linspace(0, 2 * np.pi, n_items, endpoint=False)
node_x = np.cos(angles)
node_y = np.sin(angles)

df_nodes = pd.DataFrame({
    'Category': categories,
    'X': node_x,
    'Y': node_y
})
df_nodes



,Category,X,Y
0,Bakery,1.000000,0.000000e+00
1,Dairy,0.809017,5.877853e-01
2,Produce,0.309017,9.510565e-01
3,Meat,-0.309017,9.510565e-01
4,Pantry,-0.809017,5.877853e-01
5,Snacks,-1.000000,1.224647e-16
6,Beverages,-0.809017,-5.877853e-01
7,Frozen,-0.309017,-9.510565e-01
8,Deli,0.309017,-9.510565e-01
9,Bakeware,0.809017,-5.877853e-01


## Circular Relations Network Graph

Using Plotly Scatter, we plot nodes as circles and co-purchase strengths above a threshold (0.20) as connecting paths with variable opacity.



In [3]:
fig = go.Figure()

# 1. Plot connection link lines (connections)
# Loop through lower triangle to prevent duplicates
link_count = 0
for i in range(n_items):
    for j in range(i + 1, n_items):
        weight = assoc_matrix[i, j]
        if weight >= 0.20:
            link_count += 1
            # Coordinate line
            x_line = [node_x[i], node_x[j]]
            y_line = [node_y[i], node_y[j]]
            
            fig.add_trace(go.Scatter(
                x=x_line, y=y_line,
                mode='lines',
                line=dict(color='rgba(99, 102, 241, 0.35)', width=weight * 8.0), # Width represents correlation
                showlegend=False,
                hoverinfo='none'
            ))

print(f"Rendered Connection Links: {link_count}")

# 2. Plot categories nodes
fig.add_trace(go.Scatter(
    x=list(node_x),
    y=list(node_y),
    mode='markers+text',
    marker=dict(
        size=24,
        color='#6366F1',
        line=dict(width=1.5, color='white')
    ),
    text=categories,
    textposition='top center',
    name='Product Category',
    hoverinfo='text',
    hovertext=[f"<b>{cat}</b><br>Active Associations" for cat in categories]
))

fig.update_layout(
    title='Retail Basket Co-Purchase Associations Network Matrix',
    xaxis=dict(visible=False, range=[-1.4, 1.4]),
    yaxis=dict(visible=False, range=[-1.4, 1.4]),
    width=600,
    height=550,
    template='plotly_white'
)

fig.show()


Rendered Connection Links: 41
